## SilverWork_incremental_industrial_v2

Incremental Silver processing using only new Bronze rows.

### Step 1 — Imports and setup

This cell imports Spark, Window, and Delta helpers, and creates a `silver_run_id` for the current run.


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
silver_run_id = str(uuid.uuid4())

In [0]:
print(f"Current Silver Notebook Run ID: {silver_run_id}")

### Step 2 — Silver control table

This table stores the latest Silver processing state for each entity.

It helps us track:

*   the latest Bronze run already processed by Silver
*   the latest Bronze ingestion timestamp already processed
*   how many rows were merged in the latest Silver run

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.control.object_list (
        table_id BIGINT,
        table_name STRING,
        pk_col STRING,
        layer STRING,
        join_key STRING
    )
    USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.control.processing_control (
        run_id STRING,
        table_id BIGINT,
        last_processed_at TIMESTAMP,
        rows_merged BIGINT,
        status STRING,
        updated_at TIMESTAMP
    )
    USING DELTA
""")

### Step 3: Helper Functions

In [0]:
def get_last_processed_at(table_id: int):
    processing_control_df = (
        spark.read.table("silver.control.processing_control")
        .filter((F.col("table_id") == F.lit(table_id)) & (F.col("status") == "success"))
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    last_processed_at = processing_control_df.collect()[0]["last_processed_at"]

    if not last_processed_at:
        return None
    return last_processed_at

In [0]:
def incremental_data(table_id: int, bronze_table: str):
    last_processed_at = get_last_processed_at(table_id)
    bronze_df = spark.read.table(bronze_table)

    if not last_processed_at:
        return bronze_df, last_processed_at

    incremental_df = bronze_df.filter(
        F.col("bronze_ingested_at") > F.lit(last_processed_at)
    )
    return incremental_df, last_processed_at

In [0]:
def upsert_processing_control(
    run_id: str,
    table_id: int,
    last_processed_at: datetime,
    rows_merged: int,
):
    data = [
        (
            run_id,
            table_id,
            last_processed_at,
            rows_merged,
            "success",
            datetime.utcnow(),
        ),
    ]

    schema = """
        run_id STRING,
        table_id BIGINT,
        last_processed_at TIMESTAMP,
        rows_merged BIGINT,
        status STRING, 
        updated_at TIMESTAMP
    """
    source_df = spark.createDataFrame(data, schema)
    target_tbl = DeltaTable.forName(spark, "silver.control.processing_control")

    target_tbl.alias("t").merge(
        source_df.alias("s"), "t.table_id = s.table_id"
    ).whenMatchedUpdate(
        set={
            "t.run_id": "s.run_id",
            "t.last_processed_at": "s.last_processed_at",
            "t.rows_merged": "s.rows_merged",
            "t.status": "s.status",
            "t.updated_at": "s.updated_at",
        }
    ).whenNotMatchedInsertAll().execute()

In [0]:
def load_data(source_df, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        DeltaTable.forName(spark, target_table).alias("t").merge(
            source_df.alias("s"), f"t.{join_key} = s.{join_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        source_df.write.format("delta").saveAsTable(target_table)

In [0]:
def process_cleaning(incremental_df):
    cleaned_df = incremental_df
    validated_df

In [0]:
def process_validation(cleaned_df):
    validated_df = cleaned_df
    return validated_df

In [0]:
object_list = (
    spark.read.table("silver.control.object_list")
    .filter(F.col("layer") == "silver")
    .collect()
)

for row in object_list:
    table_id = row["table_id"]
    table_name = row["table_name"]
    pk_col = row["pk_col"]
    join_key = row["join_key"]

    incremental_df, last_processed_at = incremental_data(
        table_id, f"bronze.curated.{table_name}_raw"
    )

    incremental_count = incremental_df.count()

    if incremental_count > 0:
        cleaned_df = process_cleaning(incremental_df)
        load_data(cleaned_df, f"silver.curated.{table_name}_cleaned", join_key)
        validated_df = process_validation(cleaned_df)

        correct_df = validated_df.filter(
            F.col("issue") == F.lit("No issues")
        ).withColumn("silver_run_id", F.lit(silver_run_id))

        load_data(correct_df, f"silver.curated.{table_name}_transformed", join_key)

        bad_record_df = (
            validated_df.filter(F.col("issue") != F.lit("No issues"))
            .withColumn("silver_run_id", F.lit(silver_run_id))
            .withColumn("quarantine_ts", F.current_timestamp())
        )

        bad_record_df.write.mode("append").option("mergeSchema", True).saveAsTable(
            f"silver.bad_records.{table_name}_bad_records"
        )

        max_ts = incremental_df.agg(
            F.max("bronze_ingested_at").alias("max_ts")
        ).collect()[0]["max_ts"]

        upsert_processing_control(silver_run_id, table_id, max_ts, incremental_count)

    else:
        upsert_processing_control(
            silver_run_id, table_id, last_processed_at, incremental_count
        )

![image_1774277057245.png](./image_1774277057245.png "image_1774277057245.png")

![image_1774277576850.png](./image_1774277576850.png "image_1774277576850.png")
![image_1774277598973.png](./image_1774277598973.png "image_1774277598973.png")
![image_1774277713075.png](./image_1774277713075.png "image_1774277713075.png")
![image_1774277939626.png](./image_1774277939626.png "image_1774277939626.png")
![image_1774278249043.png](./image_1774278249043.png "image_1774278249043.png")
![image_1774278289241.png](./image_1774278289241.png "image_1774278289241.png")